In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

from catboost import CatBoostClassifier

# =========================================================
# Config
# =========================================================

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"
ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "targeted_slice_features"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

TARGET = "임신 성공 여부"

BASELINE_VALID_AUC = 0.737416
POS_WEIGHT = 190123 / 66228

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("ANALYSIS_DIR:", ANALYSIS_DIR)

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True
ANALYSIS_DIR: /mnt/c/dev/my_ml_project/analysis/targeted_slice_features


In [2]:
def reset_raw_data():
    """
    매 실험마다 원본 train/test를 다시 로드해서 깨끗한 상태로 반환.
    """
    train_raw = pd.read_csv(TRAIN_PATH)
    test_raw = pd.read_csv(TEST_PATH)

    X_raw = train_raw.drop(columns=[TARGET])
    y = train_raw[TARGET].astype(int).reset_index(drop=True)
    X_test_raw = test_raw.copy()

    id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

    X_raw = X_raw.drop(columns=id_cols, errors="ignore").reset_index(drop=True)
    X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore").reset_index(drop=True)

    return X_raw, y, X_test_raw


X_raw_check, y_check, X_test_raw_check = reset_raw_data()

print("X_raw:", X_raw_check.shape)
print("y:", y_check.shape)
print("X_test_raw:", X_test_raw_check.shape)
print("target mean:", y_check.mean())

X_raw: (256351, 67)
y: (256351,)
X_test_raw: (90067, 67)
target mean: 0.2583489044318142


In [3]:
FEATURE_SETS = {
    # 기준 확인용
    "baseline": [],

    # 1단계: 가장 명확한 극단 구간
    "zero_transfer_storage": [
        "zero_transfer",
        "storage",
    ],

    # 2단계: 배아 이식 경과일만 따로
    "transfer_day": [
        "transfer_day",
    ],

    # 3단계: abs calibration gap에서 나온 이식 효율 계열
    "transfer_efficiency": [
        "transfer_efficiency",
    ],

    # 4단계: 가장 유력한 것만 결합
    "zero_storage_day": [
        "zero_transfer",
        "storage",
        "transfer_day",
    ],

    # 5단계: 전체 후보 결합
    "all_targeted": [
        "zero_transfer",
        "storage",
        "transfer_day",
        "transfer_efficiency",
    ],
}

In [4]:
def _num(df, col, fill_value=0):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(fill_value)
    return pd.Series(fill_value, index=df.index)


def _str_col(df, col, fill_value=""):
    if col in df.columns:
        return df[col].astype(str).str.strip()
    return pd.Series(fill_value, index=df.index)


def add_raw_targeted_slice_features(df, feature_keys):
    """
    data_preprocessing() 전에 raw 상태에서 만들 feature.

    raw 상태에서 만드는 이유:
    - NaN/문자열 원본 의미를 보존하기 위해
    - 배아 생성 주요 이유 같은 원본 category를 그대로 쓰기 위해
    """
    df = df.copy()

    transfer_embryo = _num(df, "이식된 배아 수")
    created_embryo = _num(df, "총 생성 배아 수")
    stored_embryo = _num(df, "저장된 배아 수")
    thawed_embryo = _num(df, "해동된 배아 수")

    embryo_reason = _str_col(df, "배아 생성 주요 이유")

    if "배아 이식 경과일" in df.columns:
        transfer_day = pd.to_numeric(df["배아 이식 경과일"], errors="coerce")
    else:
        transfer_day = pd.Series(np.nan, index=df.index)

    new_cols = []

    # =====================================================
    # 1. 이식 없음 계열
    # =====================================================
    if "zero_transfer" in feature_keys:
        df["이식배아_0_flag"] = (transfer_embryo == 0).astype(int)

        df["이식없음_생성배아있음"] = (
            (transfer_embryo == 0) &
            (created_embryo > 0)
        ).astype(int)

        df["이식없음_저장배아있음"] = (
            (transfer_embryo == 0) &
            (stored_embryo > 0)
        ).astype(int)

        df["이식없음_해동배아있음"] = (
            (transfer_embryo == 0) &
            (thawed_embryo > 0)
        ).astype(int)

        new_cols += [
            "이식배아_0_flag",
            "이식없음_생성배아있음",
            "이식없음_저장배아있음",
            "이식없음_해동배아있음",
        ]

    # =====================================================
    # 2. 배아 저장용 계열
    # =====================================================
    if "storage" in feature_keys:
        df["배아저장용_flag"] = (
            embryo_reason == "배아 저장용"
        ).astype(int)

        df["배아저장용_이식없음"] = (
            (df["배아저장용_flag"] == 1) &
            (transfer_embryo == 0)
        ).astype(int)

        df["배아저장용_저장배아있음"] = (
            (df["배아저장용_flag"] == 1) &
            (stored_embryo > 0)
        ).astype(int)

        df["배아저장용_IVF"] = (
            (df["배아저장용_flag"] == 1) &
            (_str_col(df, "시술 유형") == "IVF")
        ).astype(int)

        new_cols += [
            "배아저장용_flag",
            "배아저장용_이식없음",
            "배아저장용_저장배아있음",
            "배아저장용_IVF",
        ]

    # =====================================================
    # 3. 배아 이식 경과일 targeted flag
    # =====================================================
    if "transfer_day" in feature_keys:
        df["배아이식일_0_flag"] = (transfer_day == 0).astype(int)
        df["배아이식일_1_flag"] = (transfer_day == 1).astype(int)
        df["배아이식일_5_flag"] = (transfer_day == 5).astype(int)

        df["배아이식일_0_or_1"] = (
            transfer_day.isin([0, 1])
        ).astype(int)

        df["배아이식일_4_or_5"] = (
            transfer_day.isin([4, 5])
        ).astype(int)

        # 이식일 × 이식 배아 수
        df["이식일0_이식배아수"] = df["배아이식일_0_flag"] * transfer_embryo
        df["이식일1_이식배아수"] = df["배아이식일_1_flag"] * transfer_embryo
        df["이식일5_이식배아수"] = df["배아이식일_5_flag"] * transfer_embryo

        new_cols += [
            "배아이식일_0_flag",
            "배아이식일_1_flag",
            "배아이식일_5_flag",
            "배아이식일_0_or_1",
            "배아이식일_4_or_5",
            "이식일0_이식배아수",
            "이식일1_이식배아수",
            "이식일5_이식배아수",
        ]

    return df, new_cols


def add_post_targeted_slice_features(df, feature_keys):
    """
    data_preprocessing() 이후에 만들 feature.

    이유:
    - 배아_이식률, 배아_이식_집중도 같은 기존 파생변수를 활용하기 위해
    """
    df = df.copy()
    new_cols = []

    transfer_embryo = _num(df, "이식된 배아 수")
    created_embryo = _num(df, "총 생성 배아 수")
    transfer_day = _num(df, "배아 이식 경과일", fill_value=-999)

    embryo_transfer_rate = _num(df, "배아_이식률")
    embryo_focus = _num(df, "배아_이식_집중도")

    if "transfer_efficiency" in feature_keys:
        # abs gap 표에서 나온 후보:
        # 생성 배아는 많은데 이식률이 낮은 그룹
        df["생성많고_이식률낮음"] = (
            (created_embryo >= 5) &
            (embryo_transfer_rate <= 0.25) &
            (transfer_embryo > 0)
        ).astype(int)

        # 이식 집중도 중간 구간
        df["배아집중도_mid_flag"] = (
            (embryo_focus > 0.167) &
            (embryo_focus <= 0.667)
        ).astype(int)

        # 이식일 5일 × 배아 집중도
        df["이식일5_배아집중도"] = (
            (transfer_day == 5).astype(int) *
            embryo_focus
        )

        # 중간 집중도 + 이식일 5일
        df["배아집중도_mid_이식일5"] = (
            (df["배아집중도_mid_flag"] == 1) &
            (transfer_day == 5)
        ).astype(int)

        # 중간 집중도 + 생성 배아 많음
        df["배아집중도_mid_생성많음"] = (
            (df["배아집중도_mid_flag"] == 1) &
            (created_embryo >= 5)
        ).astype(int)

        new_cols += [
            "생성많고_이식률낮음",
            "배아집중도_mid_flag",
            "이식일5_배아집중도",
            "배아집중도_mid_이식일5",
            "배아집중도_mid_생성많음",
        ]

    return df, new_cols

In [5]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [6]:
def data_preprocessing_targeted_slice_exp(df, feature_keys):
    df = df.copy()

    # 1. raw feature 생성
    df, raw_new_cols = add_raw_targeted_slice_features(df, feature_keys)

    # 2. 기존 champion preprocessing
    df = data_preprocessing(df)

    # 3. 기존 파생변수 기반 post feature 생성
    df, post_new_cols = add_post_targeted_slice_features(df, feature_keys)

    new_cols = raw_new_cols + post_new_cols

    return df, new_cols

In [7]:
def run_single_valid_experiment(exp_name, feature_keys):
    """
    원본 데이터에서 다시 시작해서
    feature set별 CatBoost 단일 valid를 실행.
    """
    print("\n" + "=" * 100)
    print("Experiment:", exp_name)
    print("feature_keys:", feature_keys)
    print("=" * 100)

    # 매번 데이터 초기화
    X_raw, y, _ = reset_raw_data()

    # feature 생성 + preprocessing
    X_exp, new_cols = data_preprocessing_targeted_slice_exp(
        X_raw,
        feature_keys=feature_keys
    )

    print("X_exp shape:", X_exp.shape)
    print("new_cols:", new_cols)

    if new_cols:
        existing_new_cols = [col for col in new_cols if col in X_exp.columns]
        print("\nnew cols nunique:")
        print(X_exp[existing_new_cols].nunique())
        display(X_exp[existing_new_cols].head())
    else:
        existing_new_cols = []

    # train/valid split
    X_train, X_val, y_train, y_val = train_test_split(
        X_exp,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    X_train = X_train.copy()
    X_val = X_val.copy()

    cat_cols = X_train.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.02498214961001344,
        depth=8,
        l2_leaf_reg=18.591182129683194,
        random_strength=0.32969640414889206,
        bagging_temperature=4.535604806522509,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=100,
        class_weights=[1, POS_WEIGHT],
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=100
    )

    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]

    metrics = {
        "exp_name": exp_name,
        "feature_keys": ",".join(feature_keys),
        "f1": f1_score(y_val, y_val_pred),
        "precision": precision_score(y_val, y_val_pred),
        "recall": recall_score(y_val, y_val_pred),
        "roc_auc": roc_auc_score(y_val, y_val_proba),
        "baseline_auc": BASELINE_VALID_AUC,
        "diff": roc_auc_score(y_val, y_val_proba) - BASELINE_VALID_AUC,
        "best_iteration": model.best_iteration_,
        "n_features": X_exp.shape[1],
        "new_cols": ",".join(existing_new_cols),
    }

    print("\nMetrics")
    for k, v in metrics.items():
        print(k, ":", v)

    # FI
    fi = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.get_feature_importance()
    }).sort_values("importance", ascending=False)

    print("\nTop FI")
    display(fi.head(50))

    if existing_new_cols:
        print("\nNew feature FI")
        display(fi[fi["feature"].isin(existing_new_cols)])

    # 저장
    exp_dir = ANALYSIS_DIR / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    pd.DataFrame([metrics]).to_csv(exp_dir / "metrics.csv", index=False)
    fi.to_csv(exp_dir / "feature_importance.csv", index=False)

    if existing_new_cols:
        X_exp[existing_new_cols].head(1000).to_csv(
            exp_dir / "new_feature_sample.csv",
            index=False
        )

    return metrics, fi

In [8]:
all_results = []

metrics, fi = run_single_valid_experiment(
    exp_name="baseline",
    feature_keys=FEATURE_SETS["baseline"]
)

all_results.append(metrics)

result_df = pd.DataFrame(all_results)
display(result_df)


Experiment: baseline
feature_keys: []
X_exp shape: (256351, 92)
new_cols: []
0:	test: 0.7211759	best: 0.7211759 (0)	total: 263ms	remaining: 8m 44s
100:	test: 0.7340689	best: 0.7340689 (100)	total: 48.6s	remaining: 15m 12s
200:	test: 0.7361859	best: 0.7361859 (200)	total: 1m 14s	remaining: 11m 4s
300:	test: 0.7368888	best: 0.7368888 (300)	total: 1m 27s	remaining: 8m 15s
400:	test: 0.7371866	best: 0.7371866 (400)	total: 1m 40s	remaining: 6m 40s
500:	test: 0.7372815	best: 0.7372831 (489)	total: 1m 52s	remaining: 5m 35s
600:	test: 0.7373504	best: 0.7373504 (600)	total: 2m 4s	remaining: 4m 50s
700:	test: 0.7373808	best: 0.7373901 (675)	total: 2m 17s	remaining: 4m 14s
800:	test: 0.7373943	best: 0.7374160 (739)	total: 2m 30s	remaining: 3m 44s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7374159611
bestIteration = 739

Shrink model to first 740 iterations.

Metrics
exp_name : baseline
feature_keys : 
f1 : 0.515556891627837
precision : 0.3857978404319136
recall : 0.7768

,feature,importance
84,배아_이식_집중도,27.605003
41,이식된 배아 수,21.134722
52,난자 출처,6.943069
1,시술 당시 나이,4.646812
80,배아_냉동률,3.999044
83,고령_난자수_interaction,3.938497
65,배아 이식 경과일,3.591906
70,배아 이식 경과일_performed,2.562485
47,수집된 신선 난자 수,2.561089
38,총 생성 배아 수,2.544344


,exp_name,feature_keys,f1,precision,recall,roc_auc,baseline_auc,diff,best_iteration,n_features,new_cols
0,baseline,,0.515557,0.385798,0.776838,0.737416,0.737416,-3.886681e-08,739,92,


In [9]:
metrics, fi = run_single_valid_experiment(
    exp_name="zero_transfer_storage",
    feature_keys=FEATURE_SETS["zero_transfer_storage"]
)

all_results.append(metrics)

result_df = pd.DataFrame(all_results).sort_values("roc_auc", ascending=False)
display(result_df)

result_df.to_csv(ANALYSIS_DIR / "experiment_results.csv", index=False)


Experiment: zero_transfer_storage
feature_keys: ['zero_transfer', 'storage']
X_exp shape: (256351, 100)
new_cols: ['이식배아_0_flag', '이식없음_생성배아있음', '이식없음_저장배아있음', '이식없음_해동배아있음', '배아저장용_flag', '배아저장용_이식없음', '배아저장용_저장배아있음', '배아저장용_IVF']

new cols nunique:
이식배아_0_flag     2
이식없음_생성배아있음     2
이식없음_저장배아있음     2
이식없음_해동배아있음     2
배아저장용_flag      2
배아저장용_이식없음      2
배아저장용_저장배아있음    2
배아저장용_IVF       2
dtype: int64


,이식배아_0_flag,이식없음_생성배아있음,이식없음_저장배아있음,이식없음_해동배아있음,배아저장용_flag,배아저장용_이식없음,배아저장용_저장배아있음,배아저장용_IVF
0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0


0:	test: 0.7227858	best: 0.7227858 (0)	total: 173ms	remaining: 5m 45s
100:	test: 0.7338451	best: 0.7338451 (100)	total: 13.8s	remaining: 4m 20s
200:	test: 0.7360867	best: 0.7360867 (200)	total: 30.2s	remaining: 4m 30s
300:	test: 0.7367997	best: 0.7367997 (300)	total: 43.7s	remaining: 4m 6s
400:	test: 0.7370190	best: 0.7370276 (396)	total: 56s	remaining: 3m 43s
500:	test: 0.7372388	best: 0.7372388 (500)	total: 1m 7s	remaining: 3m 23s
600:	test: 0.7372981	best: 0.7373032 (597)	total: 1m 21s	remaining: 3m 9s
700:	test: 0.7373620	best: 0.7373692 (661)	total: 1m 34s	remaining: 2m 55s
800:	test: 0.7373980	best: 0.7373984 (799)	total: 1m 47s	remaining: 2m 41s
900:	test: 0.7374118	best: 0.7374404 (881)	total: 2m	remaining: 2m 27s
1000:	test: 0.7374067	best: 0.7374473 (933)	total: 2m 13s	remaining: 2m 12s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7374472846
bestIteration = 933

Shrink model to first 934 iterations.

Metrics
exp_name : zero_transfer_storage
feature_key

,feature,importance
92,배아_이식_집중도,25.416050
67,이식배아_0_flag,17.133527
41,이식된 배아 수,8.568741
52,난자 출처,6.209397
1,시술 당시 나이,5.017607
91,고령_난자수_interaction,4.300689
65,배아 이식 경과일,3.346024
47,수집된 신선 난자 수,3.129605
38,총 생성 배아 수,2.384396
43,저장된 배아 수,2.291179



New feature FI


,feature,importance
67,이식배아_0_flag,17.133527
70,이식없음_해동배아있음,0.146291
72,배아저장용_이식없음,0.000000
71,배아저장용_flag,0.000000
68,이식없음_생성배아있음,0.000000
73,배아저장용_저장배아있음,0.000000
74,배아저장용_IVF,0.000000
69,이식없음_저장배아있음,0.000000


,exp_name,feature_keys,f1,precision,recall,roc_auc,baseline_auc,diff,best_iteration,n_features,new_cols
1,zero_transfer_storage,"zero_transfer,storage",0.515668,0.386165,0.775857,0.737447,0.737416,3.128465e-05,933,100,"이식배아_0_flag,이식없음_생성배아있음,이식없음_저장배아있음,이식없음_해동배아있..."
0,baseline,,0.515557,0.385798,0.776838,0.737416,0.737416,-3.886681e-08,739,92,


In [10]:
metrics, fi = run_single_valid_experiment(
    exp_name="transfer_day",
    feature_keys=FEATURE_SETS["transfer_day"]
)

all_results.append(metrics)

result_df = pd.DataFrame(all_results).sort_values("roc_auc", ascending=False)
display(result_df)

result_df.to_csv(ANALYSIS_DIR / "experiment_results.csv", index=False)


Experiment: transfer_day
feature_keys: ['transfer_day']
X_exp shape: (256351, 100)
new_cols: ['배아이식일_0_flag', '배아이식일_1_flag', '배아이식일_5_flag', '배아이식일_0_or_1', '배아이식일_4_or_5', '이식일0_이식배아수', '이식일1_이식배아수', '이식일5_이식배아수']

new cols nunique:
배아이식일_0_flag    2
배아이식일_1_flag    2
배아이식일_5_flag    2
배아이식일_0_or_1    2
배아이식일_4_or_5    2
이식일0_이식배아수      4
이식일1_이식배아수      4
이식일5_이식배아수      4
dtype: int64


,배아이식일_0_flag,배아이식일_1_flag,배아이식일_5_flag,배아이식일_0_or_1,배아이식일_4_or_5,이식일0_이식배아수,이식일1_이식배아수,이식일5_이식배아수
0,0,0,0,0,0,0.0,0.0,0.0
1,0,0,0,0,0,0.0,0.0,0.0
2,0,0,0,0,0,0.0,0.0,0.0
3,0,0,0,0,0,0.0,0.0,0.0
4,0,0,0,0,0,0.0,0.0,0.0


0:	test: 0.7210694	best: 0.7210694 (0)	total: 159ms	remaining: 5m 18s
100:	test: 0.7345394	best: 0.7345394 (100)	total: 12.9s	remaining: 4m 2s
200:	test: 0.7365597	best: 0.7365597 (200)	total: 25.2s	remaining: 3m 45s
300:	test: 0.7371156	best: 0.7371156 (300)	total: 36.8s	remaining: 3m 27s
400:	test: 0.7374025	best: 0.7374025 (400)	total: 49s	remaining: 3m 15s
500:	test: 0.7374734	best: 0.7374734 (500)	total: 1m	remaining: 3m 2s
600:	test: 0.7375025	best: 0.7375025 (600)	total: 1m 12s	remaining: 2m 49s
700:	test: 0.7375481	best: 0.7375518 (663)	total: 1m 25s	remaining: 2m 38s
800:	test: 0.7375436	best: 0.7375674 (734)	total: 1m 37s	remaining: 2m 26s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7375674355
bestIteration = 734

Shrink model to first 735 iterations.

Metrics
exp_name : transfer_day
feature_keys : transfer_day
f1 : 0.5154887218045113
precision : 0.3858332708036317
recall : 0.7763853238713574
roc_auc : 0.7375674355390728
baseline_auc : 0.737416
diff :

,feature,importance
41,이식된 배아 수,28.242114
92,배아_이식_집중도,22.034172
52,난자 출처,7.955446
1,시술 당시 나이,4.827464
91,고령_난자수_interaction,3.837310
78,배아 이식 경과일_performed,3.565154
88,배아_냉동률,3.199359
38,총 생성 배아 수,2.334561
43,저장된 배아 수,2.131931
53,정자 출처,2.047465



New feature FI


,feature,importance
71,배아이식일_4_or_5,0.483702
69,배아이식일_5_flag,0.347755
74,이식일5_이식배아수,0.192670
67,배아이식일_0_flag,0.082538
72,이식일0_이식배아수,0.074449
68,배아이식일_1_flag,0.063225
73,이식일1_이식배아수,0.062777
70,배아이식일_0_or_1,0.025141


,exp_name,feature_keys,f1,precision,recall,roc_auc,baseline_auc,diff,best_iteration,n_features,new_cols
2,transfer_day,transfer_day,0.515489,0.385833,0.776385,0.737567,0.737416,1.514355e-04,734,100,"배아이식일_0_flag,배아이식일_1_flag,배아이식일_5_flag,배아이식일_0..."
1,zero_transfer_storage,"zero_transfer,storage",0.515668,0.386165,0.775857,0.737447,0.737416,3.128465e-05,933,100,"이식배아_0_flag,이식없음_생성배아있음,이식없음_저장배아있음,이식없음_해동배아있..."
0,baseline,,0.515557,0.385798,0.776838,0.737416,0.737416,-3.886681e-08,739,92,


In [11]:
metrics, fi = run_single_valid_experiment(
    exp_name="transfer_efficiency",
    feature_keys=FEATURE_SETS["transfer_efficiency"]
)

all_results.append(metrics)

result_df = pd.DataFrame(all_results).sort_values("roc_auc", ascending=False)
display(result_df)

result_df.to_csv(ANALYSIS_DIR / "experiment_results.csv", index=False)


Experiment: transfer_efficiency
feature_keys: ['transfer_efficiency']
X_exp shape: (256351, 97)
new_cols: ['생성많고_이식률낮음', '배아집중도_mid_flag', '이식일5_배아집중도', '배아집중도_mid_이식일5', '배아집중도_mid_생성많음']

new cols nunique:
생성많고_이식률낮음         2
배아집중도_mid_flag     2
이식일5_배아집중도        42
배아집중도_mid_이식일5     2
배아집중도_mid_생성많음     2
dtype: int64


,생성많고_이식률낮음,배아집중도_mid_flag,이식일5_배아집중도,배아집중도_mid_이식일5,배아집중도_mid_생성많음
0,0,1,0.0,0,0
1,0,0,0.0,0,0
2,0,0,0.0,0,0
3,0,0,0.0,0,0
4,0,0,0.0,0,0


0:	test: 0.7190995	best: 0.7190995 (0)	total: 143ms	remaining: 4m 45s
100:	test: 0.7338805	best: 0.7338805 (100)	total: 11.9s	remaining: 3m 43s
200:	test: 0.7360118	best: 0.7360118 (200)	total: 24.2s	remaining: 3m 36s
300:	test: 0.7366224	best: 0.7366241 (298)	total: 36.8s	remaining: 3m 27s
400:	test: 0.7368456	best: 0.7368456 (400)	total: 48.4s	remaining: 3m 12s
500:	test: 0.7369514	best: 0.7369554 (495)	total: 1m	remaining: 3m 2s
600:	test: 0.7370880	best: 0.7370885 (599)	total: 1m 13s	remaining: 2m 52s
700:	test: 0.7371970	best: 0.7372130 (684)	total: 1m 38s	remaining: 3m 3s
800:	test: 0.7373140	best: 0.7373173 (770)	total: 1m 54s	remaining: 2m 51s
900:	test: 0.7373772	best: 0.7373796 (893)	total: 2m 30s	remaining: 3m 3s
1000:	test: 0.7373708	best: 0.7373917 (912)	total: 2m 44s	remaining: 2m 43s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7373916957
bestIteration = 912

Shrink model to first 913 iterations.

Metrics
exp_name : transfer_efficiency
feature_key

,feature,importance
41,이식된 배아 수,25.866115
84,배아_이식_집중도,25.508324
52,난자 출처,6.800801
1,시술 당시 나이,4.598158
83,고령_난자수_interaction,4.142071
65,배아 이식 경과일,2.760488
80,배아_냉동률,2.672714
47,수집된 신선 난자 수,2.403836
43,저장된 배아 수,2.303584
38,총 생성 배아 수,2.145583



New feature FI


,feature,importance
92,생성많고_이식률낮음,1.135771
94,이식일5_배아집중도,0.651201
93,배아집중도_mid_flag,0.165990
96,배아집중도_mid_생성많음,0.022257
95,배아집중도_mid_이식일5,0.020260


,exp_name,feature_keys,f1,precision,recall,roc_auc,baseline_auc,diff,best_iteration,n_features,new_cols
2,transfer_day,transfer_day,0.515489,0.385833,0.776385,0.737567,0.737416,1.514355e-04,734,100,"배아이식일_0_flag,배아이식일_1_flag,배아이식일_5_flag,배아이식일_0..."
1,zero_transfer_storage,"zero_transfer,storage",0.515668,0.386165,0.775857,0.737447,0.737416,3.128465e-05,933,100,"이식배아_0_flag,이식없음_생성배아있음,이식없음_저장배아있음,이식없음_해동배아있..."
0,baseline,,0.515557,0.385798,0.776838,0.737416,0.737416,-3.886681e-08,739,92,
3,transfer_efficiency,transfer_efficiency,0.515427,0.386269,0.774347,0.737392,0.737416,-2.430431e-05,912,97,"생성많고_이식률낮음,배아집중도_mid_flag,이식일5_배아집중도,배아집중도_mid..."


In [12]:
metrics, fi = run_single_valid_experiment(
    exp_name="zero_storage_day",
    feature_keys=FEATURE_SETS["zero_storage_day"]
)

all_results.append(metrics)

result_df = pd.DataFrame(all_results).sort_values("roc_auc", ascending=False)
display(result_df)

result_df.to_csv(ANALYSIS_DIR / "experiment_results.csv", index=False)


Experiment: zero_storage_day
feature_keys: ['zero_transfer', 'storage', 'transfer_day']
X_exp shape: (256351, 108)
new_cols: ['이식배아_0_flag', '이식없음_생성배아있음', '이식없음_저장배아있음', '이식없음_해동배아있음', '배아저장용_flag', '배아저장용_이식없음', '배아저장용_저장배아있음', '배아저장용_IVF', '배아이식일_0_flag', '배아이식일_1_flag', '배아이식일_5_flag', '배아이식일_0_or_1', '배아이식일_4_or_5', '이식일0_이식배아수', '이식일1_이식배아수', '이식일5_이식배아수']

new cols nunique:
이식배아_0_flag     2
이식없음_생성배아있음     2
이식없음_저장배아있음     2
이식없음_해동배아있음     2
배아저장용_flag      2
배아저장용_이식없음      2
배아저장용_저장배아있음    2
배아저장용_IVF       2
배아이식일_0_flag    2
배아이식일_1_flag    2
배아이식일_5_flag    2
배아이식일_0_or_1    2
배아이식일_4_or_5    2
이식일0_이식배아수      4
이식일1_이식배아수      4
이식일5_이식배아수      4
dtype: int64


,이식배아_0_flag,이식없음_생성배아있음,이식없음_저장배아있음,이식없음_해동배아있음,배아저장용_flag,배아저장용_이식없음,배아저장용_저장배아있음,배아저장용_IVF,배아이식일_0_flag,배아이식일_1_flag,배아이식일_5_flag,배아이식일_0_or_1,배아이식일_4_or_5,이식일0_이식배아수,이식일1_이식배아수,이식일5_이식배아수
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0
3,1,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0


0:	test: 0.7227681	best: 0.7227681 (0)	total: 153ms	remaining: 5m 6s
100:	test: 0.7343813	best: 0.7343813 (100)	total: 13s	remaining: 4m 4s
200:	test: 0.7364685	best: 0.7364685 (200)	total: 26.8s	remaining: 4m
300:	test: 0.7370543	best: 0.7370543 (300)	total: 39.9s	remaining: 3m 45s
400:	test: 0.7372649	best: 0.7372649 (400)	total: 57.4s	remaining: 3m 49s
500:	test: 0.7373447	best: 0.7373472 (496)	total: 1m 32s	remaining: 4m 37s
600:	test: 0.7373952	best: 0.7373952 (600)	total: 1m 46s	remaining: 4m 6s
700:	test: 0.7374304	best: 0.7374390 (619)	total: 2m	remaining: 3m 42s
800:	test: 0.7374307	best: 0.7374582 (729)	total: 2m 15s	remaining: 3m 22s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7374582132
bestIteration = 729

Shrink model to first 730 iterations.

Metrics
exp_name : zero_storage_day
feature_keys : zero_transfer,storage,transfer_day
f1 : 0.5144492706319516
precision : 0.38539668209005756
recall : 0.7734410388041673
roc_auc : 0.7374582132295926
baseline

,feature,importance
100,배아_이식_집중도,22.709271
41,이식된 배아 수,16.317127
67,이식배아_0_flag,10.471459
52,난자 출처,5.804554
1,시술 당시 나이,4.743320
99,고령_난자수_interaction,4.506776
96,배아_냉동률,3.297185
43,저장된 배아 수,3.040730
38,총 생성 배아 수,2.555894
86,배아 이식 경과일_performed,2.428609



New feature FI


,feature,importance
67,이식배아_0_flag,10.471459
79,배아이식일_4_or_5,0.695172
70,이식없음_해동배아있음,0.243191
82,이식일5_이식배아수,0.234178
77,배아이식일_5_flag,0.096066
75,배아이식일_0_flag,0.063532
76,배아이식일_1_flag,0.062571
80,이식일0_이식배아수,0.060954
78,배아이식일_0_or_1,0.056129
81,이식일1_이식배아수,0.051009


,exp_name,feature_keys,f1,precision,recall,roc_auc,baseline_auc,diff,best_iteration,n_features,new_cols
2,transfer_day,transfer_day,0.515489,0.385833,0.776385,0.737567,0.737416,1.514355e-04,734,100,"배아이식일_0_flag,배아이식일_1_flag,배아이식일_5_flag,배아이식일_0..."
4,zero_storage_day,"zero_transfer,storage,transfer_day",0.514449,0.385397,0.773441,0.737458,0.737416,4.221323e-05,729,108,"이식배아_0_flag,이식없음_생성배아있음,이식없음_저장배아있음,이식없음_해동배아있..."
1,zero_transfer_storage,"zero_transfer,storage",0.515668,0.386165,0.775857,0.737447,0.737416,3.128465e-05,933,100,"이식배아_0_flag,이식없음_생성배아있음,이식없음_저장배아있음,이식없음_해동배아있..."
0,baseline,,0.515557,0.385798,0.776838,0.737416,0.737416,-3.886681e-08,739,92,
3,transfer_efficiency,transfer_efficiency,0.515427,0.386269,0.774347,0.737392,0.737416,-2.430431e-05,912,97,"생성많고_이식률낮음,배아집중도_mid_flag,이식일5_배아집중도,배아집중도_mid..."
